# Project: state tomography
**Quantum Computing · Week 3 (The maths of qubits, and interference) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qc/w3d5.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Project: state tomography. Measure many copies of a mystery state in the X, Y and Z bases,
rebuild its Bloch vector, and check how close you got.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector, state_fidelity

sampler = StatevectorSampler(seed=12)
_rng = np.random.default_rng(31)
_t, _p = np.arccos(1 - 2 * _rng.random()), 2 * np.pi * _rng.random()

def mystery():
    """A circuit that prepares a hidden one-qubit state (don't peek!)."""
    qc = QuantumCircuit(1)
    qc.ry(_t, 0)
    qc.rz(_p, 0)
    return qc

def p_first(basis, shots):
    """Probability of the first outcome (0, + or +i) when measuring the mystery state in a basis."""
    qc = mystery()
    if basis == "X":
        qc.h(0)
    elif basis == "Y":
        qc.sdg(0)
        qc.h(0)
    qc.measure_all()
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    return counts.get("0", 0) / shots

print({b: p_first(b, 100) for b in "XYZ"})

## Project: rebuild the state
1. Estimate the Bloch vector: each coordinate is $2P - 1$ in its basis.
2. If shot noise made the arrow longer than 1, shrink it to length 1.
3. Turn the arrow into angles, $\theta = \arccos z$ and $\varphi = \operatorname{atan2}(y, x)$, and into a state.
4. Compare with the truth using the fidelity, for 100, 1000 and 10000 shots per basis.

In [ ]:
def tomography(shots):
    v = np.zeros(3)          # TODO: [2 * p_first(b, shots) - 1 for b in "XYZ"], shrunk to length 1 if needed
    theta = 0.0              # TODO: from v
    phi = 0.0                # TODO: from v
    return Statevector([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])

truth = Statevector(mystery())
for shots in [100, 1000, 10000]:
    print(f"{shots:6d} shots per basis: fidelity {state_fidelity(tomography(shots), truth):.4f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def tomography(shots):
    v = np.array([2 * p_first(b, shots) - 1 for b in "XYZ"])
    length = np.linalg.norm(v)
    if length > 1:
        v = v / length
    theta = np.arccos(np.clip(v[2] / max(np.linalg.norm(v), 1e-12), -1, 1))
    phi = np.arctan2(v[1], v[0])
    return Statevector([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])

for shots in [100, 1000, 10000]:
    print(f"{shots:6d} shots per basis: fidelity {state_fidelity(tomography(shots), truth):.4f}")
print(f"\nThe hidden angles were theta = {np.degrees(_t):.1f} and phi = {np.degrees(_p):.1f} degrees.")